# Fusion training (DJA)

1. **Extract embeddings** — run the frozen image + spectrum encoders over `images/dja_crossmatch/dja_matched.h5` (image-store layout: `image`/`id`/`ra`/`dec`, `id` → DJA FITS row). Spectra (`flux`/`valid_spec`) and `sn50` are fetched from `DJA_spectra_v4.5.fits` by `id`. Writes `image_cls_embed` / `image_patch_embed` / `spectrum_patch_embed` / `spectrum_token_mask` / `n_valid` / `sn50` back into the matched H5.
2. **Train fusion** — launch `trainer.py fit --config config_dja.yaml` (reads those embeddings).

**Filters available** (FusionDataModule): `min_sn50` (→ `sn50`), `min_n_valid`/`max_n_valid` (→ `n_valid`), `frac_valid_pix` (→ `spectrum_token_mask`).

In [1]:
# ── Extract embeddings from image + spectrum encoders ─────────────────────────
# Save EVERYTHING at this step; pooling/CLS-vs-tokens is decided downstream.
#   image encoder    → CLS embedding (N,512) + patch-token embeddings (N,P,512)
#   spectrum encoder → patch-token embeddings (N,Nspec,64) + valid-token mask (N,Nspec)
# (the spectrum CLS is intentionally NOT used: no gradient reaches it in pretraining.)
#
# New crossmatch format: `dja_matched.h5` is image-store layout (image/id/ra/dec),
# where `id` is the row index into the DJA FITS. Spectra (flux/valid_spec) and the
# `sn50` quality scalar are NO LONGER embedded in the matched file — they are
# fetched from the DJA FITS by `id`. We also write `sn50` so the min_sn50 filter
# in FusionDataModule keeps working (alongside n_valid and spectrum_token_mask).
#
# Note: DataLoaders use num_workers=0 — this is a small/IO-light extraction, and
# worker processes only added the noisy "_MultiProcessingDataLoaderIter.__del__ …
# can only test a child process" teardown tracebacks on shutdown.
import os
os.environ["XFORMERS_DISABLED"] = "1"   # must be set before dinov2 is imported

import sys
from pathlib import Path

import h5py
import numpy as np
import torch
from astropy.io import fits
from astropy.table import Table
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

# the following paths are relative to the project root
PROJECT_ROOT = Path("/home/yacheng/ssl_outthere")

# New crossmatch output: image-store layout (image/id/ra/dec); `id` → DJA FITS row.
DJA_H5_crossmatched = PROJECT_ROOT / "images/dja_crossmatch/dja_matched.h5"
# Standalone embeddings file consumed by fusion + every downstream task. Holds only
# the precomputed embeddings + metadata (id/ra/dec/sn50/n_valid/spectrum_stats); the
# raw image/spectrum arrays stay in the crossmatch file and are NOT duplicated here.
EMB_H5 = PROJECT_ROOT / "encoder_fusion/embeddings.h5"

# Source DJA FITS — holds the shared wavelength grid (HDU 'WAVE') plus the full
# per-object spectra (flux/valid_spec) and quality scalars (sn50). We index it
# by the matched `id`.
DJA_FITS    = PROJECT_ROOT / "images/DJA/DJA_spectra_v4.5.fits"

# Image encoder (AstroDINO): config + teacher weights
IMG_CONFIG  = PROJECT_ROOT / "encoder_image/astrodino/model/astrodino_f150w_vitb_ps6_st3_bs128_distribute/config.yaml"
IMG_WEIGHTS = PROJECT_ROOT / "encoder_image/astrodino/model/astrodino_f150w_vitb_ps6_st3_bs128_distribute/eval/training_289999/teacher_checkpoint.pth"

# Spectrum encoder (LowResPT): Lightning checkpoint
SPEC_CKPT   = PROJECT_ROOT / "encoder_spectrum/LowResPT/outputs/low_res_pt_1_2_micron/version_1/checkpoints/epoch=epoch=267-val_hid_loss=val_hid_loss=0.1033.ckpt"

# Image data lives at a flat key in the new image-store layout.
IMG_KEY     = "image"            # (N, 128, 128) f150w cutouts

BATCH_SIZE  = 512
CROP_SIZE   = 72       # cfg.crops.global_crops_size for this image encoder
NUM_WORKERS = 0        # 0 → no DataLoader worker subprocesses (avoids teardown noise)
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
# ╚══════════════════════════════════════════════════════════════════════════╝

# Make dinov2 + preprocessing + LowResPT importable
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "encoder_image/astrodino"))
sys.path.insert(0, str(PROJECT_ROOT / "encoder_image/astrodino/benchmark"))
sys.path.insert(0, str(PROJECT_ROOT / "encoder_spectrum/LowResPT"))   # LowResPT's `model` pkg

print(f"Device : {DEVICE}")
print(f"H5 file: {DJA_H5_crossmatched}")
with h5py.File(DJA_H5_crossmatched, "r") as f:
    n_total = f[IMG_KEY].shape[0]
    dja_ids = f["id"][:].astype(np.int64)       # row index into the DJA FITS
    match_ra  = f["ra"][:]  if "ra"  in f else None
    match_dec = f["dec"][:] if "dec" in f else None
print(f"Samples: {n_total}\n")

# ── Fetch DJA spectra + sn50 for the matched objects (id-join to the FITS) ─────
print("Loading DJA spectra / sn50 by id …")
dja_tab = Table.read(DJA_FITS)                                       # CATALOG HDU
flux_m  = np.asarray(dja_tab["flux"])[dja_ids].astype(np.float32)    # (N, 473) f_nu (uJy)
valid_m = np.asarray(dja_tab["valid_spec"])[dja_ids].astype(bool)    # (N, 473) per-pixel validity
sn50_m  = np.asarray(dja_tab["sn50"])[dja_ids].astype(np.float32)    # (N,) quality scalar
# Catalog F150W total photometry → image-side absolute brightness stat (same band as
# the cutout). The fixed arcsinh stretch keeps brightness in the image tokens but
# compresses/saturates it; this is the clean, undistorted physical scale. Invalid
# entries (-99 sentinel / non-finite) → NaN; fusion's StatsEncoder imputes them.
f150_tot  = np.asarray(dja_tab["phot_f150w_tot_1"])[dja_ids].astype(np.float32)   # (N,)
f150_etot = np.asarray(dja_tab["phot_f150w_etot_1"])[dja_ids].astype(np.float32)  # (N,)
img_stats = np.stack([f150_tot, f150_etot], axis=1)                 # (N, 2)
bad = ~np.isfinite(img_stats) | (img_stats <= -90)                  # -99 sentinel
img_stats[bad.any(axis=1)] = np.nan                                 # mark whole row missing
del dja_tab
print(f"  flux {flux_m.shape}, valid {valid_m.shape}, sn50 {sn50_m.shape}, "
      f"img_stats {img_stats.shape} ({np.isfinite(img_stats[:,0]).mean()*100:.1f}% valid)")


# ── Datasets ──────────────────────────────────────────────────────────────────
class DJAImageDataset(Dataset):
    def __init__(self, h5_path, crop_size, to_rgb):
        self.h5_path = h5_path
        self.crop    = transforms.CenterCrop(crop_size)
        self.to_rgb  = to_rgb
        with h5py.File(h5_path, "r") as f:
            self.n = f[IMG_KEY].shape[0]

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        with h5py.File(self.h5_path, "r") as f:
            img = f[IMG_KEY][idx].astype("float32")        # (H, W)  MJy/sr
        img_t = torch.from_numpy(img[np.newaxis])          # (1, H, W)
        img_t = self.crop(img_t)                           # (1, 72, 72)
        img_t = torch.from_numpy(self.to_rgb(img_t.numpy()))
        return img_t


class DJASpectrumDataset(Dataset):
    """Replicates LowResPT/data/dataset.py preprocessing so the frozen encoder
    sees inputs identical to training: window the shared wave grid to
    [wl_ref_min, wl_ref_max], use the `valid_spec` flag as the validity mask,
    and (unless use_jansky) convert f_nu → f_lambda via flux / lambda**2.

    Spectra are passed in as in-memory arrays (fetched from the DJA FITS by id)."""

    def __init__(self, flux, valid, wave_win, wl_keep, use_jansky):
        self.flux       = flux                          # (N, 473) f_nu
        self.valid      = valid                         # (N, 473) bool
        self.wave_win   = wave_win.astype("float32")    # (Lwin,) shared grid
        self.wl_keep    = wl_keep                        # (473,) bool window mask
        self.use_jansky = use_jansky
        self.n          = len(flux)

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        flux  = self.flux[idx][self.wl_keep].astype("float32")    # (Lwin,) f_nu
        valid = self.valid[idx][self.wl_keep]                     # (Lwin,) bool
        finite = np.isfinite(flux)
        flux = np.where(finite, flux, 0.0)
        valid_mask = valid & finite
        if not self.use_jansky:
            flux = flux / (self.wave_win ** 2)             # f_lambda ∝ f_nu / lambda^2
        return {
            "flux":       torch.from_numpy(flux),
            "wavelength": torch.from_numpy(self.wave_win.copy()),
            "valid_mask": torch.from_numpy(valid_mask),
        }


def spec_collate(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}


@torch.no_grad()
def extract_image_embeddings(model, loader):
    """Return (cls (N,512), patch (N,P,512)) from the frozen DINOv2 teacher."""
    model.eval()
    cls_parts, patch_parts = [], []
    for imgs in tqdm(loader, desc="Image embeddings"):
        out = model.forward_features(imgs.to(DEVICE))
        cls_parts.append(out["x_norm_clstoken"].cpu().numpy())        # (B, 512)
        patch_parts.append(out["x_norm_patchtokens"].cpu().numpy())   # (B, P, 512)
    return np.concatenate(cls_parts, 0), np.concatenate(patch_parts, 0)


@torch.no_grad()
def extract_spectrum_embeddings(model, loader):
    """Return (patch (N,Nspec,64), token_mask (N,Nspec) bool, stats (N,2)) from
    LowResPT. `stats` is the per-sample [mean, std] of the (f_lambda) flux used by
    _normalize_flux — the absolute brightness scale that per-sample normalisation
    removes from the patch tokens; saved so fusion can re-inject it.
    Mirrors the encode pipeline used in linear_probe_redshift.ipynb."""
    model.eval()
    min_std = float(model.hparams.min_std)
    patch_parts, mask_parts, stats_parts = [], [], []
    for batch in tqdm(loader, desc="Spectrum embeddings"):
        flux  = batch["flux"].to(DEVICE).float()
        wave  = batch["wavelength"].to(DEVICE).float()
        vmask = batch["valid_mask"].to(DEVICE).bool()
        fn, mean, std = model._normalize_flux(flux, vmask, min_std)
        stats = torch.cat([mean, std], dim=-1)                       # (B, 2) [mean, std]
        patches, wt, vp, tvm, _ = model._patchify(fn, wave, vmask)
        ps = model._compute_patch_stats(patches, vp)
        x  = model.encode(patches, wt, tvm, stats, patch_stats=ps)   # (B, N+1, D)
        patch_parts.append(x[:, 1:, :].cpu().numpy())                # drop CLS → (B, N, D)
        mask_parts.append(tvm.cpu().numpy())                         # (B, N) valid-token mask
        stats_parts.append(stats.cpu().numpy())                      # (B, 2) abs flux scale
    return (np.concatenate(patch_parts, 0),
            np.concatenate(mask_parts, 0),
            np.concatenate(stats_parts, 0))


# ── Image encoder ─────────────────────────────────────────────────────────────
print("[1/4] Loading image encoder …")
from omegaconf import OmegaConf
from dinov2.eval.setup import build_model_for_eval
from preprocessing import get_torgb

cfg = OmegaConf.load(IMG_CONFIG)
img_model = build_model_for_eval(cfg, pretrained_weights=str(IMG_WEIGHTS)).eval().to(DEVICE)
to_rgb, in_chans = get_torgb(cfg)
print(f"  Loaded — in_chans={in_chans}, embed_dim={img_model.embed_dim}")

# ── Spectrum encoder ──────────────────────────────────────────────────────────
print("[2/4] Loading spectrum encoder …")
from model.low_res_pt import LowResPT
spec_model = LowResPT.load_from_checkpoint(str(SPEC_CKPT), map_location=DEVICE).eval().to(DEVICE)
print(f"  Loaded — embed_dim={spec_model.embed_dim}  "
      f"patch={spec_model.hparams.patch_size}  stride={spec_model.hparams.stride}")

# Replicate the training wavelength window (hparams carry the values the
# checkpoint was trained with: wl_ref_min/max, use_jansky).
hp          = spec_model.hparams
wl_ref_min  = hp.get("wl_ref_min", None)
wl_ref_max  = hp.get("wl_ref_max", None)
use_jansky  = bool(hp.get("use_jansky", False))
with fits.open(DJA_FITS, memmap=False) as hdul:
    wave_full = np.asarray(hdul["WAVE"].data, dtype=np.float32)   # (473,)
wl_keep = np.ones(wave_full.shape[0], dtype=bool)
if wl_ref_min is not None:
    wl_keep &= wave_full > wl_ref_min
if wl_ref_max is not None:
    wl_keep &= wave_full < wl_ref_max
wave_win = wave_full[wl_keep]
print(f"  Spectral window: {wave_win.shape[0]} px "
      f"({wave_win[0]:.3f}–{wave_win[-1]:.3f} µm), use_jansky={use_jansky}")

# ── Extract ───────────────────────────────────────────────────────────────────
print("[3/4] Extracting image embeddings …")
img_loader = DataLoader(DJAImageDataset(DJA_H5_crossmatched, CROP_SIZE, to_rgb),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
img_cls, img_patch = extract_image_embeddings(img_model, img_loader)
print(f"  Done — cls {img_cls.shape}, patch {img_patch.shape}")

print("[4/4] Extracting spectrum embeddings …")
spec_loader = DataLoader(DJASpectrumDataset(flux_m, valid_m, wave_win, wl_keep, use_jansky),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                         collate_fn=spec_collate)
spec_patch, spec_token_mask, spec_stats = extract_spectrum_embeddings(spec_model, spec_loader)
print(f"  Done — patch {spec_patch.shape}, token_mask {spec_token_mask.shape}, "
      f"stats {spec_stats.shape}")

# ── n_valid (valid spectral pixels per object, inside the window) ─────────────
valid_win = valid_m[:, wl_keep] & np.isfinite(flux_m[:, wl_keep])
n_valid   = valid_win.sum(axis=1).astype(np.int32)
print(f"\nn_valid: min={n_valid.min()}  median={int(np.median(n_valid))}  max={n_valid.max()}")

# ── Write the standalone embeddings file (fresh; fusion + downstream read this) ─
print(f"\nWriting embeddings → {EMB_H5}")
to_write = [
    ("image_cls_embed",      img_cls),          # (N, 512)
    ("image_patch_embed",    img_patch),        # (N, P, 512)
    ("spectrum_patch_embed", spec_patch),       # (N, Nspec, 64)
    ("spectrum_token_mask",  spec_token_mask),  # (N, Nspec) bool
    ("spectrum_stats",       spec_stats),       # (N, 2)      per-sample [mean, std] flux scale
    ("image_stats",          img_stats),         # (N, 2)      catalog [f150w_tot, f150w_etot]
    ("n_valid",              n_valid),          # (N,)        → min/max_n_valid filter
    ("sn50",                 sn50_m),           # (N,)        → min_sn50 filter
    ("id",                   dja_ids),          # (N,)        row index into the DJA FITS
]
if match_ra  is not None: to_write.append(("ra",  match_ra))   # (N,) carried for downstream join
if match_dec is not None: to_write.append(("dec", match_dec))  # (N,)
with h5py.File(EMB_H5, "w") as f:
    for key, data in to_write:
        f.create_dataset(key, data=data, compression="gzip")
        print(f"  {key}: {data.shape}  {data.dtype}")
print("\nDone.")

Device : cuda
H5 file: /home/yacheng/ssl_outthere/images/dja_crossmatch/dja_matched.h5
Samples: 2666

Loading DJA spectra / sn50 by id …
  flux (2666, 473), valid (2666, 473), sn50 (2666,), img_stats (2666, 2) (96.9% valid)
[1/4] Loading image encoder …


/home/yacheng/ssl_outthere/.pixi/envs/h100/lib/python3.11/site-packages/dinov2/layers/swiglu_ffn.py:45: UserWarning: xFormers is disabled (SwiGLU)
  warnings.warn("xFormers is disabled (SwiGLU)")
/home/yacheng/ssl_outthere/.pixi/envs/h100/lib/python3.11/site-packages/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/yacheng/ssl_outthere/.pixi/envs/h100/lib/python3.11/site-packages/dinov2/layers/attention.py:29: UserWarning: xFormers is disabled (Attention)
  warnings.warn("xFormers is disabled (Attention)")
/home/yacheng/ssl_outthere/.pixi/envs/h100/lib/python3.11/site-packages/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/yacheng/ssl_outthere/.pixi/envs/h100/lib/python3.11/site-packages/dinov2/layers/block.py:35: UserWarning: xFormers is disabled (Block)
  warnings.warn("xFormers is disabled (Block)

  Loaded — in_chans=1, embed_dim=512
[2/4] Loading spectrum encoder …
  Loaded — embed_dim=128  patch=4  stride=2
  Spectral window: 56 px (1.004–1.991 µm), use_jansky=False
[3/4] Extracting image embeddings …


Image embeddings:   0%|          | 0/6 [00:00<?, ?it/s]

  Done — cls (2666, 512), patch (2666, 529, 512)
[4/4] Extracting spectrum embeddings …


Spectrum embeddings:   0%|          | 0/6 [00:00<?, ?it/s]

  Done — patch (2666, 27, 128), token_mask (2666, 27), stats (2666, 2)

n_valid: min=0  median=56  max=56

Writing embeddings → /home/yacheng/ssl_outthere/encoder_fusion/embeddings.h5
  image_cls_embed: (2666, 512)  float32
  image_patch_embed: (2666, 529, 512)  float32
  spectrum_patch_embed: (2666, 27, 128)  float32
  spectrum_token_mask: (2666, 27)  bool
  spectrum_stats: (2666, 2)  float32
  image_stats: (2666, 2)  float32
  n_valid: (2666,)  int32
  sn50: (2666,)  float32
  id: (2666,)  int64
  ra: (2666,)  float64
  dec: (2666,)  float64

Done.


## Train fusion

Reads `image_embed` / `spectrum_embed` from the H5 above (paths/keys set in `config_dja.yaml`).

In [2]:
!LD_LIBRARY_PATH=/home/yacheng/ssl_outthere/.pixi/envs/h100/lib:$LD_LIBRARY_PATH XFORMERS_DISABLED=1 python trainer.py fit --config config_dja.yaml

Seed set to 42
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Missing logger folder: outputs/fusion_dja_image_attn_spectrum_attn
[FusionDataModule] selection: 2115/2666 samples kept
FusionEmbeddingDataset: 1903 samples, mask_prob=0.05, coverage: {'image': 1903, 'spectrum': 1903}
FusionEmbeddingDataset: 212 samples, mask_prob=0.00, coverage: {'image': 212, 'spectrum': 212}
[setup] Registered modality 'image' with auto-detected input_dim=512, pool=attention, stats_dim=2
[setup] Registered modality 'spectrum' with auto-detected input_dim=128, pool=attention, stats_dim=2
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
/home/yacheng/ssl_outthere/.pixi/envs/h100/lib/python3.11/site-packages/lightning/pytorch/loops/fit_loop.py:298: The number of training batches (3) is smaller than the logging interval Trainer(log_every_n